In [ ]:
import duckdb 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 

In [2]:
#db connection

base_parquet_dir = "../data/interim/jobposts_base.parquet"

con = duckdb.connect()
con.read_parquet(base_parquet_dir).create_view("jobposts");

In [8]:
jobposts = con.sql(""" 
    select *
    from jobposts
""").df()

display(jobposts.head())

,RowNumber,Jobpost_RawTitle,Jobpost_PreferredGender,Jobpost_ProvinceFa,Jobpost_ProvinceEn,Jobpost_WorkTypeFa,Jobpost_WorkTypeEn,Jobpost_SalaryCanBeShown,Jobpost_MinSalary,Jobpost_MaxSalary,...,Company_ActivityTypeFa,Company_ActivityTypeEn,Company_IndustryFa,Company_IndustryEn,date,year,month,jalali_year,jalali_month,jalali_season
0,1,مهندس مکانیک,تفاوتی ندارد,تهران,Tehran,پاره وقت,Part Time,False,NaN,NaN,...,شرکت ایرانی دارای مشتریان داخلی,Iranian company dealing only with Iranian enti...,"ساختمان / مصالح و تجهیزات ساختمانی,تولیدی / صنعتی","Construction / Building Materials & Equipment,...",2022-04-01,2022,4,1401,1,Spring
1,2,طراح و گرافیست,تفاوتی ندارد,تهران,Tehran,تمام وقت,Full Time,False,NaN,NaN,...,شرکت ایرانی دارای مشتریان داخلی و خارجی,Iranian company dealing with Iranian and forei...,طراحی / گرافیک / هنر,Design / Arts,2023-03-01,2023,3,1401,12,Winter
2,3,برنامه نویس full-stack,تفاوتی ندارد,خراسان رضوی,Khorasan Razavi,تمام وقت,Full Time,False,NaN,NaN,...,شرکت ایرانی دارای مشتریان داخلی و خارجی,Iranian company dealing with Iranian and forei...,"اینترنت / تجارت الکترونیک / خدمات آنلاین,شرکت ...",Internet Provider / E-commerce / Online Servic...,2022-04-01,2022,4,1401,1,Spring
3,4,بازاریاب - خانم,فقط خانم,البرز,Alborz,تمام وقت,Full Time,False,NaN,NaN,...,شرکت ایرانی دارای مشتریان داخلی,Iranian company dealing only with Iranian enti...,"اینترنت / تجارت الکترونیک / خدمات آنلاین,خرده ...",Internet Provider / E-commerce / Online Servic...,2023-12-01,2023,12,1402,9,Autumn
4,5,برنامه نویس React Native,تفاوتی ندارد,اصفهان,Isfahan,پاره وقت,Part Time,False,NaN,NaN,...,شرکت ایرانی دارای مشتریان داخلی و خارجی,Iranian company dealing with Iranian and forei...,فناوری اطلاعات / نرم افزار و سخت افزار,IT / Software / Hardware,2022-07-01,2022,7,1401,4,Summer


### Parse the Education Column

In [11]:
#parse Jobpost_AcademicFields column
from src.parsers import safe_json_loads

res = jobposts["Jobpost_AcademicFields"].map(safe_json_loads)
jobposts["_edu_parsed"] = res.map(lambda t: t[0])   # list of dicts or None
jobposts["_edu_status"] = res.map(lambda t: t[1])   # ok / missing / failed

print(jobposts["_edu_status"].value_counts())
print(jobposts["_edu_status"].value_counts(normalize=True).round(4))

# Failure rate among rows that actually had a value
had_value = jobposts["_edu_status"] != "missing"
fail_rate = (jobposts["_edu_status"] == "failed").sum() / had_value.sum()
print(f"\nParse failure rate (non-missing rows): {fail_rate:.2%}")


_edu_status
missing    193088
ok         171750
Name: count, dtype: int64
_edu_status
missing    0.5292
ok         0.4708
Name: proportion, dtype: float64

Parse failure rate (non-missing rows): 0.00%
